In [1]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(ggpubr)
    library(BSgenome)
    library(biomaRt)
    library(BSgenome.Ocuniculus.NCBI.oryCun2)
    library(GenomicFeatures)
    library(compEpiTools)
    library(GenomeInfoDb)
    library(stringi)
    library(AnnotationHub)
})


groupGOTerms: 	GOBPTerm, GOMFTerm, GOCCTerm environments built.

No methods found in package 'BiocGenerics' for requests: 'clusterApplyLB', 'clusterEvalQ' when loading 'methylPipe'

No methods found in package 'BiocGenerics' for requests: 'clusterApplyLB', 'clusterEvalQ' when loading 'compEpiTools'



In [2]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/0X_Rabbit_ATAC/'
io$output.directory <- file.path(io$basedir,"ArchR")
setwd(io$output.directory)

In [56]:
opts = list()
# Options
opts$min.fragments <- 2500
opts$filterTSS.score <- 3 # May need to rerun with threshold at 2

# ArchR options
addArchRThreads(threads = 1) 

Setting default number of Parallel threads to 1.



In [57]:
# Create genome annotation
genomeAnnotation <- createGenomeAnnotation(genome = BSgenome.Ocuniculus.NCBI.oryCun2)

#important as rabbit chromosome dont have the 'chr' prefix
addArchRChrPrefix(chrPrefix = FALSE)

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

ArchR is now disabling the requirement of chromosome prefix = 'chr'



In [58]:
# Create a gene annotation using Txdb file
rabbit_txdb <- makeTxDbFromBiomart(biomart="ensembl", dataset="ocuniculus_gene_ensembl")

Warning message:
"Ensembl will soon enforce the use of https.
Ensure the 'host' argument includes "https://""
Download and preprocess the 'transcripts' data frame ... 
OK

Download and preprocess the 'chrominfo' data frame ... 
OK

Download and preprocess the 'splicings' data frame ... 
OK

Download and preprocess the 'genes' data frame ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
OK



In [59]:
taxid = taxonomyId(rabbit_txdb)
## If we don't have a package, then lets get the taxIds and AHIds
        ## for the hub objects
        loadNamespace("AnnotationHub")
        ah <- AnnotationHub::AnnotationHub()
        ah <- subset(ah, ah$rdataclass=='OrgDb') 
        mc <- mcols(ah)[,'taxonomyid', drop=FALSE]
        ## Then just get the object
        AHID <- rownames(mc[mc$taxonomyid==taxid,,drop=FALSE])
        rabbit_OrgDb <- ah[[AHID[2]]]

<environment: namespace:AnnotationHub>

snapshotDate(): 2021-10-20

loading from cache



In [60]:
#rabbit_txdb
geneAnnotation <- createGeneAnnotation(
  TxDb  = rabbit_txdb,
    OrgDb =rabbit_OrgDb,
  TSS = TSS(rabbit_txdb), 
  exons = exons(rabbit_txdb)
)

'select()' returned 1:1 mapping between keys and columns

Getting Genes..

Determined Annotation Style = ENSEMBL

Getting Exons..

Getting TSS..



In [3]:
# ensembl = useMart("ensembl")
# ensembl = useDataset("ocuniculus_gene_ensembl", mart = ensembl)

# gene_map = getBM(attributes=c("ensembl_gene_id", "external_gene_name"), mart = ensembl)
# gene_map[gene_map$external_gene_name=='',] = gene_map[gene_map$external_gene_name=='',]$ensembl_gene_id


# gene_map = gene_map[match(geneAnnotation$genes$gene_id, gene_map$ensembl_gene_id),]
# geneAnnotation$genes$symbol = gene_map$external_gene_name

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'match': object 'geneAnnotation' not found


In [5]:
write.csv(gene_map, paste0(io$basedir, 'gene_map.csv'), row.names=FALSE)

In [62]:
seqlevelsStyle(genomeAnnotation$chromSizes) <- 'NCBI'

In [63]:
genomeAnnotation

List of length 3
names(3): genome chromSizes blacklist

In [64]:
addArchRThreads(1)

Setting default number of Parallel threads to 1.



In [66]:
fragment_files = list.files(paste0(io$basedir, 'data/'))

In [ ]:
#Create Arrow File for filtered samples, filtered using Signac's filter for ATAC
ArrowFiles <- createArrowFiles(
  inputFiles = paste0(io$basedir, 'data/', fragment_files),
  sampleNames = paste0('rabbit_', strsplit(fragment_files,"_") %>% map_chr(1)),
  minTSS = opts$filterTSS.score, #Dont set this too high because you can always increase later
  minFrags = opts$min.fragments , 
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)